# ???????? NN Back Propagation

?Notebook???Andrew Ng \"Machine Leanring\"?[????](https://github.com/loveunk/machine-learning-deep-learning-notes/blob/master/machine-learning/neural-networks.md)???????

???python???3.6

* ??????:https://github.com/loveunk/machine-learning-deep-learning-notes/
* ????????????,???????????:www.kaikai.ai

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import matplotlib
import scipy.optimize as opt
from sklearn.metrics import classification_report#????????

In [ ]:
def load_data(path, transpose=True):
    data = sio.loadmat(path)
    y = data.get('y')  # (5000,1)
    y = y.reshape(y.shape[0])

    X = data.get('X')  # (5000,400)

    if transpose:
        X = np.array([im.reshape((20, 20)).T for im in X])
        X = np.array([im.reshape(400) for im in X])

    return X, y

In [ ]:
X, _ = load_data('ex4data1.mat')

## ?????(10*10???)

In [ ]:
def plot_100_image(X):
    """ sample 100 image and show them
    assume the image is square

    X : (5000, 400)
    """
    size = int(np.sqrt(X.shape[1]))

    # sample 100 image, reshape, reorg it
    sample_idx = np.random.choice(np.arange(X.shape[0]), 100)  # 100*400
    sample_images = X[sample_idx, :]

    fig, ax_array = plt.subplots(nrows=10, ncols=10, sharey=True, sharex=True, figsize=(8, 8))

    for r in range(10):
        for c in range(10):
            ax_array[r, c].matshow(sample_images[10 * r + c].reshape((size, size)),
                                   cmap=matplotlib.cm.binary)
            plt.xticks(np.array([]))
            plt.yticks(np.array([]))

In [ ]:
plot_100_image(X)
plt.show()

# ????

In [ ]:
X_raw, y_raw = load_data('ex4data1.mat', transpose=False)

# ?????1???,
X = np.insert(X_raw, 0, np.ones(X_raw.shape[0]), axis=1)
X.shape

In [ ]:
y_raw

In [ ]:
def expand_y(y):
#     """expand 5000*1 into 5000*10
#     where y=10 -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]: ndarray
#     """
    res = []
    for i in y:
        y_array = np.zeros(10)
        y_array[i - 1] = 1

        res.append(y_array)

    return np.array(res)
# from sklearn.preprocessing import OneHotEncoder
# encoder = OneHotEncoder(sparse=False)
# y_onehot = encoder.fit_transform(y)
# y_onehot.shape #?????expand_y(y)??

In [ ]:
y = expand_y(y_raw)
y

# ????

In [ ]:
def load_weight(path):
    data = sio.loadmat(path)
    return data['Theta1'], data['Theta2']

In [ ]:
t1, t2 = load_weight('ex4weights.mat')
t1.shape, t2.shape

???2??
???nn???,???theta1(25,401),theta2(10,26),??????delta1,delta2  

In [ ]:
def serialize(a, b):
    return np.concatenate((np.ravel(a), np.ravel(b)))

In [ ]:
theta = serialize(t1, t2)  # ?????,25*401+10*26=10285
theta.shape

# ???? Feedforward
> (400 + 1) -> (25 + 1) -> (10)

<img style="float: center;" src="./img/nn_model.png">

In [ ]:
def feed_forward(theta, X):
    """apply to architecture 400+1 * 25+1 *10
    X: 5000 * 401
    """

    t1, t2 = deserialize(theta)  # t1: (25,401) t2: (10,26)
    m = X.shape[0]
    a1 = X  # 5000 * 401

    z2 = a1 @ t1.T  # 5000 * 25
    a2 = np.insert(sigmoid(z2), 0, np.ones(m), axis=1)  # 5000*26

    z3 = a2 @ t2.T  # 5000 * 10
    h = sigmoid(z3)  # 5000*10, this is h_theta(X)

    # return?????????????
    return a1, z2, a2, z3, h

Sigmoid??????:
$$g(z)={\frac{1}{1+e^{{-z}}}}$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def deserialize(seq):
#     """into ndarray of (25, 401), (10, 26)"""
    return seq[:25 * 401].reshape(25, 401), seq[25 * 401:].reshape(10, 26)


In [ ]:
_, _, _, _, h = feed_forward(theta, X)
h # 5000*10

# ????
?????,????????????:
$$J(\Theta)=\frac{1}{m}\sum\limits_{i=1}^m\sum\limits_{k=1}^K{\left[{-y_k^{(i)}}\log\left({h_\Theta}\left({x^{(i)}}\right)\right)_k-\left(1-{y_k^{(i)}}\right)\log\left(1-{h_\theta}\left({x^{(i)}}\right)\right)_k\right]}$$

??????,$y$ ? $h_{\theta} \in R^{5000 \times 10}$?

????m?k??,???????? $y*log(h_{\theta}) - (1-y)*log(1-h_{\theta})$  
??????????????????????

In [ ]:
def cost(theta, X, y):
    """
    y: (m, k) ndarray
    """
    m = X.shape[0]  # get the data size m

    _, _, _, _, h = feed_forward(theta, X)

    # np.multiply is pairwise operation
    pair_computation = -np.multiply(y, np.log(h)) - np.multiply((1 - y), np.log(1 - h))

    return pair_computation.sum() / m


In [ ]:
cost(theta, X, y)

# ???????

??????????????:
\begin{align*}
J(\Theta)=&\frac{1}{m}\sum\limits_{i=1}^m\sum\limits_{k=1}^K{\left[{-y_k^{(i)}}\log\left({h_\Theta}\left({x^{(i)}}\right)\right)_k-\left(1-{y_k^{(i)}}\right)\log\left(1-{h_\theta}\left({x^{(i)}}\right)\right)_k\right]}\\&+\frac{\lambda}{2m}\sum\limits_{l=1}^{L-1}\sum\limits_{i=1}^{s_l}\sum\limits_{j=1}^{s_{l+1}}{\left(\Theta_{ji}^{(l)}\right)^2}
\end{align*}

In [ ]:
def regularized_cost(theta, X, y, lambd=1):
    """the first column of t1 and t2 is intercept theta, ignore them when you do regularization"""
    t1, t2 = deserialize(theta)  # t1: (25,401) t2: (10,26)
    m = X.shape[0]

    reg_t1 = (lambd / (2 * m)) * np.power(t1[:, 1:], 2).sum()  # this is how you ignore first col
    reg_t2 = (lambd / (2 * m)) * np.power(t2[:, 1:], 2).sum()

    return cost(theta, X, y) + reg_t1 + reg_t2

In [ ]:
regularized_cost(theta, X, y, lambd=1)

# ???? Backpropagation

????????????????

In [ ]:
X.shape,y.shape

In [ ]:
t1.shape, t2.shape

In [ ]:
theta.shape

## Sigmoid????
Sigmoid???????????:
$$g'(z) = {\frac{\partial}{\partial z}}g(z)=g(z)(1-g(z))$$

???????????:

In [ ]:
def sigmoid_gradient(z):
    return np.multiply(sigmoid(z), 1 - sigmoid(z))

In [ ]:
sigmoid_gradient(0)

# $\Theta$ ???


???????????????????

????????????????????????,????????????????????????????
???????:
$$\delta^{\left(3\right)}=a^{(3)}-y$$
$$\delta^{\left(2\right)}=\left(?^{(2)}\right)^T\delta^{(3)}.*g'(z^{(2)})$$
$$\Delta^{\left(l\right)}=\Delta^{(l)}+\delta^{(l+1)}\left(a^{(l)}\right)^T$$
$$\frac{\partial}{\partial\Theta_{ij}^{(l)}}J(\Theta)=\frac{1}{m}\Delta_{ij}^{l}$$

In [ ]:
def gradient(theta, X, y):
    # initialize
    t1, t2 = deserialize(theta)  # t1: (25,401) t2: (10,26)
    m = X.shape[0]

    delta1 = np.zeros(t1.shape)  # (25, 401)
    delta2 = np.zeros(t2.shape)  # (10, 26)

    # ????
    a1, z2, a2, z3, h = feed_forward(theta, X)

    for i in range(m):
        a1i = a1[i, :]  # (1, 401)
        z2i = z2[i, :]  # (1, 25)
        a2i = a2[i, :]  # (1, 26)

        hi = h[i, :]    # (1, 10)
        yi = y[i, :]    # (1, 10)

        # ?3?
        d3i = hi - yi  # (1, 10)

        # ?2?
        z2i = np.insert(z2i, 0, np.ones(1))  # make it (1, 26) to compute d2i
        d2i = np.multiply(t2.T @ d3i, sigmoid_gradient(z2i))  # (1, 26)

        # careful with np vector transpose
        delta2 += np.matrix(d3i).T @ np.matrix(a2i)  # (1, 10).T @ (1, 26) -> (10, 26)
        delta1 += np.matrix(d2i[1:]).T @ np.matrix(a1i)  # (1, 25).T @ (1, 401) -> (25, 401)

    delta1 = delta1 / m
    delta2 = delta2 / m

    return serialize(delta1, delta2)

In [ ]:
d1, d2 = deserialize(gradient(theta, X, y))

In [ ]:
d1.shape, d2.shape

# ????
???????$f_{i}(\theta)$??$\frac{\partial}{\partial\theta_{i}}J(\theta)$,?????????????$f_i$????????

$$f_{i}(\theta)\approx\dfrac{J\left(\theta^{i+}\right)-J\left(\theta^{i-}\right)}{2\varepsilon}$$

????????:

In [ ]:
def gradient_checking(theta, X, y, epsilon, regularized=False):
    def a_numeric_grad(plus, minus, regularized=False):
        """calculate a partial gradient with respect to 1 theta"""
        if regularized:
            return (regularized_cost(plus, X, y) - regularized_cost(minus, X, y)) / (epsilon * 2)
        else:
            return (cost(plus, X, y) - cost(minus, X, y)) / (epsilon * 2)

    theta_matrix = expand_array(theta)  # expand to (10285, 10285)
    epsilon_matrix = np.identity(len(theta)) * epsilon

    plus_matrix = theta_matrix + epsilon_matrix
    minus_matrix = theta_matrix - epsilon_matrix

    # calculate numerical gradient with respect to all theta
    numeric_grad = np.array([a_numeric_grad(plus_matrix[i], minus_matrix[i], regularized)
                                    for i in range(len(theta))])

    # analytical grad will depend on if you want it to be regularized or not
    analytic_grad = regularized_gradient(theta, X, y) if regularized else gradient(theta, X, y)

    # If you have a correct implementation, and assuming you used EPSILON = 0.0001
    # the diff below should be less than 1e-9
    # this is how original matlab code do gradient checking
    diff = np.linalg.norm(numeric_grad - analytic_grad) / np.linalg.norm(numeric_grad + analytic_grad)

    print('?????????????,??????? 10e-9 (?? epsilon=0.0001).\n???: {}\n'.format(diff))

In [ ]:
def expand_array(arr):
    """replicate array into matrix
    [1, 2, 3]

    [[1, 2, 3],
     [1, 2, 3],
     [1, 2, 3]]
    """
    # turn matrix back to ndarray
    return np.array(np.matrix(np.ones(arr.shape[0])).T @ np.matrix(arr))

In [ ]:
gradient_checking(theta, X, y, epsilon= 0.0001)#??????,????

# ??????

??????????????????,??????????? 

\begin{align*}
\frac{\partial}{\partial\Theta_{ij}^{(l)}}J(\Theta)&=\frac{1}{m}\Delta_{ij}^{l} \text{ , for } j = 0\\
\frac{\partial}{\partial\Theta_{ij}^{(l)}}J(\Theta)&=\frac{1}{m}\Delta_{ij}^{l} + \frac{\lambda}{m}\Theta^{(l)}_{ij}\text{ , for } j \ge 1
\end{align*}

??????????

In [ ]:
def regularized_gradient(theta, X, y, l=1):
    """don't regularize theta of bias terms"""
    m = X.shape[0]
    delta1, delta2 = deserialize(gradient(theta, X, y))
    t1, t2 = deserialize(theta)

    t1[:, 0] = 0
    reg_term_d1 = (l / m) * t1
    delta1 = delta1 + reg_term_d1

    t2[:, 0] = 0
    reg_term_d2 = (l / m) * t2
    delta2 = delta2 + reg_term_d2

    return serialize(delta1, delta2)

In [ ]:
# ????,????
gradient_checking(theta, X, y, epsilon=0.0001, regularized=True)

# ????

????????????????

In [ ]:
def random_init(size):
    return np.random.uniform(-0.12, 0.12, size)

In [ ]:
def nn_training(X, y):

    init_theta = random_init(10285)  # 25*401 + 10*26

    res = opt.minimize(fun=regularized_cost,
                       x0=init_theta,
                       args=(X, y, 1),
                       method='TNC',
                       jac=regularized_gradient,
                       options={'maxiter': 400})
    return res

In [ ]:
# ????,????
res = nn_training(X, y)
res

# ?????

In [ ]:
_, y_answer = load_data('ex4data1.mat')

In [ ]:
final_theta = res.x

In [ ]:
def show_accuracy(theta, X, y):
    _, _, _, _, h = feed_forward(theta, X)

    y_pred = np.argmax(h, axis=1) + 1

    print(classification_report(y, y_pred))
    
    correct = [1 if a == b else 0 for (a, b) in zip(y_pred, y)]
    accuracy = (sum(map(int, correct)) / float(len(correct)))
    print ('??????:{0}%'.format(accuracy * 100))

In [ ]:
show_accuracy(final_theta, X, y_answer)

# ?????

In [ ]:
def plot_hidden_layer(theta):
    """
    theta: (10285, )
    """
    final_theta1, _ = deserialize(theta)
    hidden_layer = final_theta1[:, 1:]  # ger rid of bias term theta

    fig, ax_array = plt.subplots(nrows=5, ncols=5, sharey=True, sharex=True, figsize=(5, 5))

    for r in range(5):
        for c in range(5):
            ax_array[r, c].matshow(hidden_layer[5 * r + c].reshape((20, 20)),
                                   cmap=matplotlib.cm.binary)
            plt.xticks(np.array([]))
            plt.yticks(np.array([]))

In [ ]:
plot_hidden_layer(final_theta)
plt.show()